In [1]:
#================================================================================
#ECOLOGICAL METRICS & SHANNON DIVERSITY INDEX: A MATHEMATICAL PRIMER
#================================================================================

#1. WHAT IS THE SHANNON DIVERSITY INDEX (H')?
#The Shannon Diversity Index is a popular ecological metric used to quantify the 
#diversity of species in a community. Unlike simple "Species Richness" (which 
#just counts the number of species), the Shannon Index accounts for both 
#Abundance and Evenness. A higher number indicates a more diverse and equally 
#distributed community.

#The true formula is: 
#    H' = - ∑ (p_i * ln(p_i))
    
#Where:
#    - S = Total number of species in the community (Richness).
#    - p_i = The proportion of the entire community made up of species "i" 
#            (Calculated as: individuals of species "i" / total individuals).
#    - ln = The natural logarithm.

#2. HOW WE APPLY IT TO THIS SUMMARY DATA (H_max):
#Because this specific dataset provides daily summaries (e.g., "12 species caught 
#today") rather than individual bird species breakdowns (e.g., "5 Robins, 7 Sparrows"), 
#we cannot calculate the exact p_i for the true Shannon formula. 

#Instead, this dashboard calculates the MAXIMUM POTENTIAL SHANNON DIVERSITY (H_max). 
#This metric measures what the diversity index would be if every species caught was 
#perfectly evenly distributed in abundance. 

#The formula for H_max is:
#    H_max = ln(S)

#In this code, we use our 'Cumulative_Species_Detections' (the sum of all daily 
#species counts for the year/season) as our 'S' value. Therefore, the code 
#calculates diversity as: np.log(Cumulative_Species_Detections).
#================================================================================


In [2]:
#import nessary libraries 
import pandas as pd
import numpy as np
import plotly.graph_objects as go

C:\Users\kmmci\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\kmmci\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [3]:
#Load the data 
file_path = 'summary sheets_2017-2025_03232026_proofed.xlsx'

#update to new file

df = pd.read_excel(file_path, sheet_name='summary sheets')

#Read in banding record data
df2 = pd.read_excel('BandingRecords.xlsx')

In [9]:
df2.columns.tolist()

['BRautoID',
 'KitNum',
 'RecorderID',
 'BanderID',
 'BandingCode',
 'BandNumber',
 'SpeciesCode',
 'AOUNum',
 'Age',
 'HowAge1',
 'HowAge2',
 'Sex',
 'HowSex1',
 'HowSex2',
 'Fat',
 'BroodPatch',
 'CloacalProtuberance',
 'Skull',
 'BodyMolt',
 'WingMolt',
 'PrimaryWear',
 'WingLength',
 'Weight',
 'CaptureDate',
 'Location',
 'CaptureTime',
 'TrapSite',
 'TrapType',
 'Status',
 'InjuryCode',
 'Notes',
 'SameDayRecapTime',
 'SameDayRecapTrapSite',
 'SameDayRecapNotes',
 'TailLength',
 'CrownLength',
 'CrownFigureID',
 'CrownFigureValue',
 'BackFigureID',
 'BackFigureValue',
 'UppertailCovertFigureID',
 'UppertailCovertFigureValue',
 'TailFigureID',
 'TailFigureValue',
 'PlumagePrimaryCovertID',
 'PlumageGreaterCovertID',
 'PlumagePrimariesID',
 'PlumageSecondariesID',
 'PlumageRectriciesID',
 'LeftTop',
 'LeftTopMarkerType',
 'LeftBottom',
 'LeftBottomMarkerType',
 'RightTop',
 'RightTopMarkerType',
 'RightBottom',
 'RightBottomMarkerType',
 'ReadableColorLeg',
 'ReadableNum',
 'Readab

In [4]:
#Ensure Correct Data Type
# Clean numeric columns
df['Spec_clean'] = pd.to_numeric(df['Spec'], errors='coerce')
df['Proc_clean'] = pd.to_numeric(df['Proc'], errors='coerce')
df['Ntotal hrs_clean'] = pd.to_numeric(df['Ntotal hrs'], errors='coerce')

In [5]:
# =============================================================================
# SHANNON DIVERSITY INDEX PREPARATION & CALCULATION PLAN
# -----------------------------------------------------------------------------
# 1. PREPARATION (Inside this function):
#    Since the dataset only provides the *number* of species caught per day 
#    rather than distinct species identities, we cannot calculate a true Shannon 
#    Index (H'). Instead, this function calculates a proxy for Total Species 
#    Richness (S) by summing all daily species counts into the variable 
#    'Cumulative_Species_Detections'.
#
# 2. CALCULATION (In the dashboard script):
#    Using the 'Cumulative_Species_Detections' (S) generated here, the later 
#    dashboard code calculates the Maximum Potential Shannon Diversity (H_max). 
#    It applies the mathematical formula H_max = ln(S) using NumPy's natural log: 
#    df_dash['Max_Shannon_Diversity'] = np.log(df_dash['Cumulative_Species_Detections'])
# =============================================================================


In [6]:
def compute_yearly_hours(group):
    # Filter out missing net hours and keep only days when nets were actually open
    v = group.dropna(subset=['Ntotal hrs_clean'])
    v = v[v['Ntotal hrs_clean'] > 0] 
    
    if len(v) == 0:
        return pd.Series({'Total_Net_Hours': np.nan})
        
    # Add up all the hours the nets were open over the entire time period
    total_hrs = v['Ntotal hrs_clean'].sum()
    
    return pd.Series({'Total_Net_Hours': round(total_hrs, 2)})

# --- ANALYSIS: SEASONAL NET HOURS ---
# 1. Generate your primary annual seasonal dataset using the helper function
annual_seasonal = df.groupby(['Year', 'Season']).apply(compute_yearly_hours).reset_index()
annual_seasonal = annual_seasonal.dropna(subset=['Total_Net_Hours'])

# 2. Extract the quick-access net hours lookup table directly from your cleaned data
net_hours_lookup = annual_seasonal.copy()

C:\Users\kmmci\AppData\Local\Temp\ipykernel_14808\3770752511.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  annual_seasonal = df.groupby(['Year', 'Season']).apply(compute_yearly_hours).reset_index()


In [15]:
# ==============================================================================
# STEP 1: PARSE AND PREPARE PER-SPECIES & COMBINED DAILY METRICS
# ==============================================================================
df2['CaptureDate'] = pd.to_datetime(df2['CaptureDate'])
df2['Year'] = df2['CaptureDate'].dt.year
df2['SpeciesCode'] = df2['SpeciesCode'].astype(str).str.strip()

def assign_excel_season(date):
    m, d = date.month, date.day
    if m == 11 or m == 12 or m == 1 or (m == 2 and d <= 15): return "Winter"
    elif m == 3 or (m == 4 and d <= 15): return "Spring Migration"
    elif (m == 8 and d >= 15) or m == 9 or (m == 10 and d <= 15): return "Fall Migration"
    elif m == 5 or m == 6 or m == 7 or (m == 8 and d < 15): return "Breeding Season"
    return "Other"

df2['Season'] = df2['CaptureDate'].apply(assign_excel_season)

# Dataset A: Segmented by Day, Season, and Specific Species
raw_sp = df2.dropna(subset=['SpeciesCode', 'CaptureDate', 'BandNumber']).groupby(['CaptureDate', 'Season', 'SpeciesCode']).agg(
    Daily_Proc=('BandNumber', 'nunique'),
    Daily_Spec=('SpeciesCode', 'nunique')
).reset_index()

# Dataset B: Segmented by Day and Season, Combined across ALL Species
raw_all = df2.dropna(subset=['SpeciesCode', 'CaptureDate', 'BandNumber']).groupby(['CaptureDate', 'Season']).agg(
    Daily_Proc=('BandNumber', 'nunique'),
    Daily_Spec=('SpeciesCode', 'nunique')
).reset_index()
raw_all['SpeciesCode'] = 'All'

# Combine individual species slices with the unified "All" baseline slice
daily_combined = pd.concat([raw_sp, raw_all], ignore_index=True)
daily_combined['Year'] = daily_combined['CaptureDate'].dt.year


# ==============================================================================
# STEP 2: MULTI-INDEX AGGREGATION & NET HOURS LOOKUP MATCHING
# ==============================================================================
net_hours_lookup = annual_seasonal.groupby(['Year', 'Season'])['Total_Net_Hours'].sum().reset_index()

def compute_matrix_metrics(group):
    if len(group) == 0:
        return pd.Series({'Total_Birds': np.nan, 'Total_Net_Hours': np.nan, 'CPUE (Relative Abundance)': np.nan, 'Peak_Richness': np.nan})
    
    total_proc = group['Daily_Proc'].sum()
    peak_spec = group['Daily_Spec'].max()
    
    current_year = group['Year'].iloc[0]
    current_season = group['Season'].iloc[0]
    
    match = net_hours_lookup[(net_hours_lookup['Year'] == current_year) & 
                             (net_hours_lookup['Season'].astype(str).str.strip() == str(current_season).strip())]
    total_hrs = match['Total_Net_Hours'].sum() if not match.empty else 0
    cpue = total_proc / total_hrs if total_hrs > 0 else np.nan

    return pd.Series({
        'Total_Birds': int(total_proc),
        'Total_Net_Hours': round(total_hrs, 2),
        'CPUE (Relative Abundance)': round(cpue, 3) if not pd.isna(cpue) else np.nan,
        'Peak_Richness': int(peak_spec)
    })

# Compute the global multi-index dimensional slice matrix
df_matrix = daily_combined.groupby(['Year', 'Season', 'SpeciesCode']).apply(compute_matrix_metrics).reset_index()

# Generate overarching "All Seasons (Overall Year)" rollups across the dimensional blocks
df_overall = daily_combined.groupby(['Year', 'SpeciesCode']).apply(compute_matrix_metrics).reset_index()
df_overall['Season'] = 'All Seasons (Overall Year)'

df_dash = pd.concat([df_matrix, df_overall], ignore_index=True)
df_dash['Season'] = df_dash['Season'].astype(str).str.strip()


# ==============================================================================
# STEP 3: ASSEMBLE PLOTLY CANVAS WITH INDEPENDENT FILTER ENGINES
# ==============================================================================
fig = go.Figure()

seasons_list = ['All Seasons (Overall Year)', 'Other', 'Spring Migration', 'Fall Migration', 'Breeding Season', 'Winter']
species_list = ['All'] + sorted([sp for sp in df_dash['SpeciesCode'].unique() if sp != 'All'])

metrics = {
    'CPUE (Relative Abundance)': 'Relative Abundance (Birds per Net-Hour)',
    'Total_Birds': 'Total Unique Individuals Captured',
    'Peak_Richness': 'Peak Species Richness'
}

# Add exactly ONE trace for every possible permutation to the canvas map
trace_metadata = []
for s in seasons_list:
    for sp in species_list:
        df_subset = df_dash[(df_dash['Season'] == s) & (df_dash['SpeciesCode'] == sp)].sort_values('Year')
        
        x_data = df_subset['Year'].tolist() if not df_subset.empty else []
        y_data = df_subset['CPUE (Relative Abundance)'].tolist() if not df_subset.empty else []
        custom_data = np.stack((df_subset['Total_Birds'], df_subset['Total_Net_Hours']), axis=-1) if not df_subset.empty else []
        
        # Default starting state: Visible only if 'All Seasons' AND 'All' species are active
        is_visible = (s == 'All Seasons (Overall Year)' and sp == 'All')
        
        fig.add_trace(go.Scatter(
            x=x_data, y=y_data, mode='lines+markers', name=f"{sp} ({s})", visible=is_visible, customdata=custom_data,
            hovertemplate=(
                f"<b>Year:</b> %{{x}}<br><b>Species:</b> {sp}<br><b>Season:</b> {s}<br>"
                f"<b>Value:</b> %{{y}}<br><b>Birds (Unique):</b> %{{customdata[0]}}<br>"
                f"<b>Net Hours:</b> %{{customdata[1]}} hrs<extra></extra>"
            ),
            line=dict(width=3), marker=dict(size=8)
        ))
        trace_metadata.append({'Season': s, 'SpeciesCode': sp})

# TRACKING STATE INTERFACE CONTROLS
current_season = 'All Seasons (Overall Year)'
current_species = 'All'

def generate_visibility_mask(target_season, target_species):
    return [meta['Season'] == target_season and meta['SpeciesCode'] == target_species for meta in trace_metadata]

# 1. METRIC DROPDOWN
metric_buttons = []
for m_key, m_title in metrics.items():
    y_updates = []
    for meta in trace_metadata:
        df_subset = df_dash[(df_dash['Season'] == meta['Season']) & (df_dash['SpeciesCode'] == meta['SpeciesCode'])].sort_values('Year')
        y_updates.append(df_subset[m_key].tolist() if not df_subset.empty else [])
    metric_buttons.append(dict(
        label=m_key.replace('_', ' ').replace(' (Relative Abundance)', ''),
        method="restyle", args=[{"y": y_updates}, {"yaxis.title.text": m_title}]
    ))

# 2. SPECIES DROPDOWN
species_buttons = []
for sp in species_list:
    species_buttons.append(dict(
        label=f"Species: {sp}", method="restyle",
        args=[{"visible": [meta['SpeciesCode'] == sp and meta['Season'] == current_season for meta in trace_metadata]}]
    ))

# 3. SEASON DROPDOWN
season_buttons = []
for s in seasons_list:
    season_buttons.append(dict(
        label=s, method="restyle",
        args=[{"visible": [meta['Season'] == s and meta['SpeciesCode'] == current_species for meta in trace_metadata]}]
    ))

# Adjust menu coordinates to make space for three independent menus
fig.update_layout(
    title="Multi-Dimensional Ecological Metrics Dashboard",
    yaxis_title=metrics['CPUE (Relative Abundance)'],
    xaxis=dict(title="Year", tickmode='linear', dtick=1, rangeslider=dict(visible=True, thickness=0.06)),
    template="plotly_white", margin=dict(t=180),
    updatemenus=[
        dict(active=0, buttons=metric_buttons, direction="down", x=0.02, xanchor="left", y=1.18, yanchor="top"),
        dict(active=0, buttons=species_buttons, direction="down", x=0.36, xanchor="left", y=1.18, yanchor="top"),
        dict(active=0, buttons=season_buttons, direction="down", x=0.70, xanchor="left", y=1.18, yanchor="top")
    ]
)

fig.add_annotation(x=0.02, y=1.28, xref="paper", yref="paper", text="<b>1. Metric:</b>", showarrow=False, font=dict(size=12), xanchor="left")
fig.add_annotation(x=0.36, y=1.28, xref="paper", yref="paper", text="<b>2. Species:</b>", showarrow=False, font=dict(size=12), xanchor="left")
fig.add_annotation(x=0.70, y=1.28, xref="paper", yref="paper", text="<b>3. Season Boundaries:</b>", showarrow=False, font=dict(size=12), xanchor="left")

print("Launching 3-way multi-filtered independent dashboard...")
fig.show()

C:\Users\kmmci\AppData\Local\Temp\ipykernel_14808\1708753954.py:64: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\kmmci\AppData\Local\Temp\ipykernel_14808\1708753954.py:67: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Launching 3-way multi-filtered independent dashboard...


In [16]:
# ==============================================================================
# STEP 4: COMPUTE COMBINED STRATIFIED EXPORTS FOR SYSTEM BACKUP
# ==============================================================================

# Extract the fully structured tracking tables used to build the dashboard visual layers
annual_species_seasonal_matrix = df_dash[df_dash['Season'] != 'All Seasons (Overall Year)'].copy()
annual_species_overall_matrix = df_dash[df_dash['Season'] == 'All Seasons (Overall Year)'].copy()

# Save structural tracking configurations down to local CSV records
annual_species_overall_matrix.to_csv('annual_species_summary_overall.csv', index=False)
annual_species_seasonal_matrix.to_csv('annual_species_summary_seasonal.csv', index=False)

# Display tabular summaries directly via runtime console outputs
print("=== SPECIFIC INDIVIDUAL OVERALL METRICS MATRIX (ALL & INDIVIDUAL SPECIES) ===")
print(annual_species_overall_matrix.sort_values(['Year', 'SpeciesCode']).head(20).to_string(index=False))

print("\n=== SPECIFIC INDIVIDUAL SEASONAL METRICS MATRIX (ALL & INDIVIDUAL SPECIES) ===")
print(annual_species_seasonal_matrix.sort_values(['Year', 'Season', 'SpeciesCode']).head(20).to_string(index=False))

=== SPECIFIC INDIVIDUAL OVERALL METRICS MATRIX (ALL & INDIVIDUAL SPECIES) ===
 Year                     Season SpeciesCode  Total_Birds  Total_Net_Hours  CPUE (Relative Abundance)  Peak_Richness
 2016 All Seasons (Overall Year)         All         12.0             0.00                        NaN            3.0
 2016 All Seasons (Overall Year)        GCSP          1.0             0.00                        NaN            1.0
 2016 All Seasons (Overall Year)        ORJU          8.0             0.00                        NaN            1.0
 2016 All Seasons (Overall Year)        RCKI          3.0             0.00                        NaN            1.0
 2017 All Seasons (Overall Year)        ACWO          1.0           271.75                      0.004            1.0
 2017 All Seasons (Overall Year)        AMRO          3.0            93.25                      0.032            1.0
 2017 All Seasons (Overall Year)        ATFL          2.0           271.75                      0.007  